# Six-lead CLBBB / ILBBB baseline
Local exploration of the trained Dataset 1 model. Leads: I, aVL, V1, V2, V5, V6. CLBBB is primary; ILBBB is secondary. Dataset 1 test is a subset of Dataset 2 test.

Open this notebook from the baseline directory after installing the packages in README.md. No GitHub operations are performed.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
RESULTS = Path("results")
DATA = Path("../processed")


## Configuration and training history

In [ ]:
display(json.loads((RESULTS / "config.json").read_text()))
history = pd.DataFrame(json.loads((RESULTS / "history.json").read_text()))
display(history)
history.plot(x="epoch", y=["train_loss", "val_loss"], ylabel="Weighted BCE")

## Held-out evaluation
The same frozen checkpoint and fixed threshold 0.50 are used throughout. Shared test records have the same predictions. ILBBB has only 8 test positives. Sigmoid outputs are model scores, not calibrated clinical probabilities.

In [ ]:
metrics = pd.read_csv(RESULTS / "metrics.csv")
display(metrics[metrics.population.isin(["dataset1_test", "dataset2_test"])])
if (RESULTS / "test_curves.png").exists():
    display(Image(filename=str(RESULTS / "test_curves.png")))
else:
    print("Generate plots locally with plot_results.py after evaluation.")

## Inspect individual predictions
Do not tune thresholds or choose models using these test predictions. Use validation for all such decisions.

In [ ]:
if (RESULTS / "dataset2_test_predictions.csv").exists():
    predictions = pd.read_csv(RESULTS / "dataset2_test_predictions.csv")
    display(predictions.head())
    display(predictions[predictions.ILBBB.eq(1)])
else:
    print("Record-level predictions remain local. Run evaluate.py to generate them.")


## Optional: read a six-lead test ECG
The processed arrays are available in the original workspace. They are intentionally excluded from the portable code/results ZIP.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
if (DATA / "X_dataset2_test.npy").exists():
    x = np.load(DATA / "X_dataset2_test.npy", mmap_mode="r")
    idx = 0
    leads = ["I", "aVL", "V1", "V2", "V5", "V6"]
    fig, axs = plt.subplots(6, 1, figsize=(12, 8), sharex=True)
    for i, lead in enumerate(leads):
        axs[i].plot(np.arange(1000) / 100, x[idx, i])
        axs[i].set_ylabel(lead + " (mV)")
    axs[-1].set_xlabel("Seconds")
    fig.tight_layout()
else:
    print("Prepare waveforms with prepare.py to enable this cell.")

## Reproduce locally
Use the commands in README.md to prepare data, train into a fresh results directory, and evaluate. The scripts never upload or commit files. `test_methodology.py` checks mixed-diagnosis inclusion, independent outcomes, patient leakage rejection, and architecture parity when the reference source is present.